<a href="https://colab.research.google.com/github/Polon-Lubvi/ICTIS/blob/main/01_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Установка

In [3]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets evaluate rouge_score bert_score mlflow pandera gradio sumy datasketch natasha
!pip freeze | grep -iE "^(transformers|peft|trl|bitsandbytes|accelerate|datasets|evaluate|rouge.score|bert.score|mlflow|pandera|gradio|sumy|datasketch|natasha|torch)=="

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
accelerate==1.15.0
bert-score==0.3.13
bitsandbytes==0.50.2
datasets==4.8.5
datasketch==2.0.0
evaluate==0.4.6
gradio==6.31.0
mlflow==3.18.0
natasha==1.6.0
pandera==0.36.0
peft==0.21.2
rouge_score==0.1.2
sumy==0.14.0
torch==2.14.1+cu130
transformers==5.22.1
trl==1.18.2


In [4]:
import os, time, csv, datetime
import numpy as np, pandas as pd, torch
from transformers import set_seed

SEED = 42
set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")

GPU: Tesla T4


In [6]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [7]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/tabd26"
for sub in ["data/raw", "data/processed", "checkpoints", "mlruns", "reports"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)

Mounted at /content/drive


## Журнал прогона

In [8]:
LOG = f"{ROOT}/reports/run_log.csv"

def log_step(step, started, rows_in=None, rows_out=None, skipped=False):
    new = not os.path.exists(LOG)
    with open(LOG, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["step", "started", "seconds", "rows_in", "rows_out", "skipped"])
        w.writerow([step, datetime.datetime.fromtimestamp(started).isoformat(timespec="seconds"),
                    round(time.time() - started, 1), rows_in, rows_out, skipped])

## Загрузка данных

In [9]:
from datasets import load_dataset

t0 = time.time()
raw = {}
skipped = True
for split in ["train", "validation", "test"]:
    path = f"{ROOT}/data/raw/gazeta_{split}.parquet"
    if not os.path.exists(path):
        load_dataset("IlyaGusev/gazeta", revision="refs/convert/parquet", split=split).to_parquet(path)
        skipped = False
    raw[split] = pd.read_parquet(path)
    print(split, raw[split].shape)
log_step("load_raw", t0, rows_out=sum(len(d) for d in raw.values()), skipped=skipped)

train (60964, 5)
validation (6369, 5)
test (6793, 5)


## Проверка качества данных

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa

schema = pa.DataFrameSchema(
    {
        "text":    pa.Column(str, [pa.Check.str_length(200, 20000)], nullable=False),
        "summary": pa.Column(str, [pa.Check.str_length(min_value=20)], nullable=False),
        "title":   pa.Column(str, nullable=True),
        "date":    pa.Column(str, nullable=False),
        "url":     pa.Column(str, unique=True, nullable=False),
    },
    checks=[
        pa.Check(lambda df: df["summary"].str.len() < df["text"].str.len(), name="summary_shorter_than_text"),
        pa.Check(lambda df: pd.to_datetime(df["date"], errors="coerce").notna(), name="date_parses"),
    ],
    strict=True,
)

t0 = time.time()
reports = []
for split, df in raw.items():
    try:
        schema.validate(df, lazy=True)
        print(split, "— нарушений нет")
    except pa.errors.SchemaErrors as e:
        fc = e.failure_cases.assign(split=split)
        reports.append(fc)
        print(split, "— нарушений:", len(fc))
quality = pd.concat(reports) if reports else pd.DataFrame()
quality.to_csv(f"{ROOT}/reports/quality.csv", index=False)
log_step("quality", t0, rows_in=sum(len(d) for d in raw.values()), rows_out=len(quality))
quality.groupby(["split", "check"]).size() if len(quality) else "Все правила выполнены"

## Проверка утечки между train и test

In [ ]:
from datasketch import MinHash, MinHashLSH

def shingles(text, k=5):
    w = text.lower().split()
    return {" ".join(w[i:i+k]) for i in range(max(1, len(w) - k + 1))}

def minhash(text):
    m = MinHash(num_perm=64, seed=SEED)
    for s in shingles(text):
        m.update(s.encode("utf-8"))
    return m

t0 = time.time()
train, test = raw["train"], raw["test"]
exact_url  = len(set(train.url) & set(test.url))
exact_text = len(set(train.text) & set(test.text))

lsh = MinHashLSH(threshold=0.8, num_perm=64)
for i, t in enumerate(train.text):
    lsh.insert(str(i), minhash(t))
near = [j for j, t in enumerate(test.text) if lsh.query(minhash(t))]

dates = {s: pd.to_datetime(d.date, errors="coerce") for s, d in raw.items()}
leak = {
    "exact_url": exact_url,
    "exact_text": exact_text,
    "near_duplicates_test": len(near),
    "train_dates": f"{dates['train'].min().date()} — {dates['train'].max().date()}",
    "test_dates":  f"{dates['test'].min().date()} — {dates['test'].max().date()}",
}
pd.Series(leak).to_csv(f"{ROOT}/reports/leakage.csv")
log_step("leakage", t0, rows_in=len(train) + len(test), rows_out=len(near))
leak

## Выборка для оценки

In [ ]:
EVAL_PATH = f"{ROOT}/data/processed/eval_sample.parquet"
if not os.path.exists(EVAL_PATH):
    raw["test"].sample(500, random_state=SEED).to_parquet(EVAL_PATH)
eval_df = pd.read_parquet(EVAL_PATH)
eval_df.shape

## Lead-3 и ROUGE

In [ ]:
import re, evaluate
from razdel import sentenize

def lead3(text):
    return " ".join(s.text for s in list(sentenize(text))[:3])

rouge = evaluate.load("rouge")
tokenize = lambda s: re.findall(r"\w+", s.lower())

preds = [lead3(t) for t in eval_df.text]
scores = rouge.compute(predictions=preds, references=list(eval_df.summary), tokenizer=tokenize)
scores

## Метрики

In [ ]:
METRICS = f"{ROOT}/reports/metrics.csv"

def save_result(method, preds, seconds):
    s = rouge.compute(predictions=preds, references=list(eval_df.summary), tokenizer=tokenize)
    row = {"method": method, **{k: round(float(v), 4) for k, v in s.items()},
           "sec_per_doc": round(seconds / len(preds), 3), "n": len(preds)}
    m = pd.read_csv(METRICS) if os.path.exists(METRICS) and os.path.getsize(METRICS) > 1 else pd.DataFrame(columns=["method"])
    m = pd.concat([m[m.method != method], pd.DataFrame([row])], ignore_index=True)
    m.to_csv(METRICS, index=False)
    eval_df.assign(pred=preds).to_parquet(f"{ROOT}/data/processed/preds_{method}.parquet")
    return m

t0 = time.time()
save_result("lead3", [lead3(t) for t in eval_df.text], time.time() - t0)

## TextRank

In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def textrank(text, n=3):
    sents = [s.text for s in sentenize(text)]
    if len(sents) <= n:
        return " ".join(sents)
    sim = cosine_similarity(TfidfVectorizer(token_pattern=r"\w+").fit_transform(sents))
    np.fill_diagonal(sim, 0)
    rank = nx.pagerank(nx.from_numpy_array(sim))
    top = sorted(sorted(rank, key=rank.get, reverse=True)[:n])
    return " ".join(sents[i] for i in top)

t0 = time.time()
save_result("textrank", [textrank(t) for t in eval_df.text], time.time() - t0)
log_step("baselines_extractive", t0, rows_in=len(eval_df))

## Zero-shot Qwen

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
PROMPT = "Кратко перескажи новость в 1–3 предложениях.\n\n{text}"

def build(text):
    msgs = [{"role": "user", "content": PROMPT.format(text=text[:6000])}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(texts, bs=8, max_new=160):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok([build(t) for t in texts[i:i+bs]], return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
        out += tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in out]

t0 = time.time()
print(save_result("zero_shot", generate(list(eval_df.text)), time.time() - t0))
log_step("baseline_zero_shot", t0, rows_in=len(eval_df))

## Данные для обучения

In [11]:
import mlflow, shutil
from datasets import Dataset
from transformers import BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

del model; torch.cuda.empty_cache()

MLFLOW_DB = "/content/mlflow.db"
if os.path.exists(f"{ROOT}/mlruns/mlflow.db") and not os.path.exists(MLFLOW_DB):
    shutil.copy(f"{ROOT}/mlruns/mlflow.db", MLFLOW_DB)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{MLFLOW_DB}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "gazeta_lora"

tr_df = raw["train"]
tr_df = tr_df[tr_df.text.str.len().between(200, 20000) & (tr_df.summary.str.len() < tr_df.text.str.len())]

def to_chat(df):
    return Dataset.from_list([
        {"prompt": [{"role": "user", "content": PROMPT.format(text=t[:3000])}],
         "completion": [{"role": "assistant", "content": s}]}
        for t, s in zip(df.text, df.summary)])

train_ds = to_chat(tr_df.sample(2000, random_state=SEED))
val_ds = to_chat(raw["validation"].sample(200, random_state=SEED))

## Обучение LoRA

In [19]:
ATTN = ["q_proj", "k_proj", "v_proj", "o_proj"]
CONFIGS = {
    "qlora_r8_attn":  dict(r=8,  target=ATTN, quant=True),
    "qlora_r16_attn": dict(r=16, target=ATTN, quant=True),
    "qlora_r16_all":  dict(r=16, target="all-linear", quant=True),
    "lora_r16_attn":  dict(r=16, target=ATTN, quant=False),
}

def train(name):
    mlflow.end_run()
    c, out = CONFIGS[name], f"{ROOT}/checkpoints/{name}"
    if os.path.exists(f"{out}/final/adapter_config.json"):
        return print(name, "уже обучена")
    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16) if c["quant"] else None
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16,
                                                quantization_config=quant, device_map="cuda")
    args = SFTConfig(output_dir=out, run_name=name, num_train_epochs=1,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5,
        fp16=True, gradient_checkpointing=True, max_length=1536,
        logging_steps=5, eval_strategy="steps", eval_steps=25,
        save_steps=25, save_total_limit=1, report_to="mlflow", seed=SEED)
    lora = LoraConfig(r=c["r"], lora_alpha=2 * c["r"], lora_dropout=0.05,
                      target_modules=c["target"], task_type="CAUSAL_LM")
    tr = SFTTrainer(model=base, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                    processing_class=tok, peft_config=lora)
    for p in tr.model.parameters():
        if p.requires_grad: p.data = p.data.float()
    print("примеров в обучении:", len(tr.train_dataset))
    tr.train(resume_from_checkpoint=get_last_checkpoint(out) if os.path.isdir(out) else None)
    tr.save_model(f"{out}/final")
    print(name, f"{(time.time() - t0) / 60:.1f} мин,",
          f"пик памяти GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ")
    shutil.copy(MLFLOW_DB, f"{ROOT}/mlruns/mlflow.db")
    log_step(f"train_{name}", t0, rows_in=len(train_ds))
    del tr, base; torch.cuda.empty_cache()

train("qlora_r16_attn")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

... (2 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

примеров в обучении: 2000


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,1.304606,1.143466,1.194197,0.734301,471908.000000
75,1.165883,1.133248,1.157048,0.735743,942448.000000
100,1.226996,1.131431,1.157601,0.736300,1416971.000000
125,1.279004,1.130914,1.157543,0.736259,1893230.000000


qlora_r16_attn 28.2 мин, пик памяти GPU 4.0 ГБ
